# LLM-aided OCR

Inspiration for prompts: https://github.com/Dicklesworthstone/llm_aided_ocr/blob/main/llm_aided_ocr.py#L467

In [ ]:
%load_ext autoreload 
%autoreload 2

import os

import torch
from dotenv import load_dotenv
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
)

load_dotenv()

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
)

token = os.environ["HF_TOKEN"]

# model_name = "google/gemma-7b-it"
# model_name = "meta-llama/Llama-3.1-8B"
model_name = "deepseek-ai/deepseek-llm-7b-chat"

tokenizer = AutoTokenizer.from_pretrained(model_name, token=token)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map="auto",
    quantization_config=quantization_config,
    token=token,
)

In [ ]:
import torch


def query_model(prompt, max_new_tokens):
    input_ids = tokenizer.encode(prompt, return_tensors="pt").to("cuda")
    output = model.generate(input_ids, max_new_tokens=max_new_tokens)
    response = tokenizer.decode(output[0], skip_special_tokens=True)

    return response


def build_query(prompt, input_sent) -> str:
    return prompt + "\nInput: " + input_sent


def check_semantic_validity(prompt, input_sent):
    query = build_query(prompt, input_sent)
    response = query_model(query, max_new_tokens=2)

    return response[len(query) :]


quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
)

In [ ]:
import pandas as pd

df = pd.read_json("../data/labelled/whole_pdf_docs.jsonl", lines=True)

In [ ]:
chunk = df.iloc[-58].text
chunk

In [ ]:
tmp = df["text"].head(500).str.cat(sep="\n")
tmp

In [ ]:
from tqdm.notebook import tqdm

corrections = []
for i in tqdm(range(df.shape[0])):
    ocr_correction_prompt = f"""Correggi gli errori causati dall’OCR seguendo queste linee guida:

1. Correggi i refusi e gli errori dell’OCR:
  - Correggi le parole divise a cavallo di un’interruzione di riga
  - Correggi gli errori comuni dell’OCR (ad esempio, “rn” letto come “m”)
  - Usa il contesto e il buon senso per correggere gli errori
  - Correggi solo errori evidenti, senza modificare inutilmente il contenuto
  - Non aggiungere punti o altra punteggiatura non necessaria

2. Mantieni la struttura originale:
  - Mantieni intatti tutti i titoli e sottotitoli

3. Preserva il contenuto originale:
  - Mantieni tutte le informazioni importanti presenti nel testo originale
  - Non aggiungere informazioni nuove non presenti nel testo originale
  - Rimuovi interruzioni di riga non necessarie all’interno di frasi o paragrafi
  - Mantieni le interruzioni di paragrafo

4. Mantieni la coerenza:
  - Assicurati che il contenuto si colleghi in modo fluido al contesto precedente
  - Gestisci in modo appropriato il testo che inizia o finisce a metà frase

IMPORTANTE: Rispondi SOLO con il testo corretto. Mantieni tutta la formattazione originale, incluse le interruzioni di riga. Non includere alcuna introduzione, spiegazione o metadati. 

IMPORTANTISSIMO: Non scrivere ulteriormente dopo il testo corretto.

Testo correntemente sotto processo:
{df.iloc[i].text}

Testo corretto:
"""
    correction = query_model(ocr_correction_prompt, 1000)
    corrections.append(correction[len(ocr_correction_prompt) :])

    if i == 100:
        break


In [ ]:
corrections